# Chapter 5 — Sampling: Choosing from a Probability Distribution
**Series: LM from First Principles** | Prerequisites: Chapters 1, 4A/4B

---

> **Capsule summary.** The trained model produces a probability distribution over the vocabulary at every generation step. This chapter asks: how should we pick the next token from that distribution? Five strategies are built from scratch — greedy, temperature, top-k, top-p (nucleus), and min-p — each first demonstrated on a hand-crafted 62-token vocabulary where every number is visible, then transferred to a real bigram model trained on tinyshakespeare. The chapter closes by showing how temperature and truncation compose into a single decoding pipeline and when each strategy is the right choice.

---

Chapter 4B gave us a tokenizer that controls *what* the model sees. This chapter asks what happens *after* the model makes its prediction.

At every generation step the model outputs a vector of logits — one score per vocabulary token. We apply softmax to turn those into probabilities. Then we must pick one token. That choice is sampling, and how we make it substantially affects what the model produces.

The chapter has two parts:

- **Part 1** — we build a toy vocabulary of 62 characters and hand-craft a logit vector. Every strategy is explained here, where every number is visible and interpretable.
- **Part 2** — we transfer the same strategy functions unchanged to a real bigram model trained on tinyshakespeare and observe how they behave on a learned distribution.

Open [`ch05_sampling.ipynb`](https://github.com/asbassan/AmarLMFP/blob/main/ch05_sampling.ipynb) and follow along.

---
## Part 1 — A Controlled Example

Before connecting any of this to a trained model, we build a minimal vocabulary and hand-craft a logit vector. Every number is under our control. The strategies we build here transfer unchanged to Part 2.

### Vocabulary and logits

A **logit vector** is what the model outputs at each generation step: one score per vocabulary token, representing how likely that token is to come next. A real model produces a different logit vector for every context — one for what follows `'T'`, a different one for what follows `'S'`, and so on.

Here we hand-craft **a single logit vector** for a 62-token vocabulary (a–z, A–Z, 0–9). The values are chosen to mimic plausible English-letter frequencies — common letters like `'e'` and `'t'` get high scores, digits and uppercase get low scores. This lets us study every sampling strategy on a distribution where every number is visible and interpretable, before connecting to a trained model in Part 2.

In [1]:
# input : hand-crafted character set and logit values
# output: stoi / itos / toy_logits  ready for all Part 1 strategy cells
import torch
import torch.nn.functional as F
import math

# Vocabulary: a-z + A-Z + 0-9  (62 tokens)
import string
vocab_chars = list(string.ascii_lowercase + string.ascii_uppercase + string.digits)
vocab_size  = len(vocab_chars)                      # 62
stoi = {ch: i for i, ch in enumerate(vocab_chars)}
itos = {i: ch for i, ch in enumerate(vocab_chars)}

print(f"Vocabulary : {vocab_size} tokens")
print(f"  Lowercase: {list(string.ascii_lowercase)}")
print(f"  Uppercase: {list(string.ascii_uppercase)}")
print(f"  Digits   : {list(string.digits)}")

# Hand-crafted logits — designed to give a realistic English-like distribution.
# High logits for common English letters; low for uppercase; very low for digits.
logit_values = {
    'e': 4.0, 't': 3.2, 'a': 2.8, 'o': 2.3, 'i': 2.0,
    'n': 1.7, 's': 1.4, 'h': 1.1, 'r': 0.8, 'l': 0.5,
    'd': 0.2, 'u': 0.0, 'c': -0.2, 'm': -0.4, 'f': -0.6,
    'w': -0.8, 'g': -1.0, 'p': -1.2, 'y': -1.4, 'b': -1.6,
    'v': -1.8, 'k': -2.0, 'j': -2.5, 'x': -3.0, 'q': -3.5, 'z': -4.0,
}
toy_logits = torch.tensor(
    [logit_values.get(ch, -3.0) for ch in vocab_chars],
    dtype=torch.float32
)

print()
print(f"Logit range: {toy_logits.min().item():.1f}  to  {toy_logits.max().item():.1f}")
print("Top 5 by logit:")
top5 = torch.topk(toy_logits, 5)
for val, idx in zip(top5.values, top5.indices):
    print(f"  '{vocab_chars[idx.item()]}' : {val.item():.1f}")

C:\Users\asbas\AppData\Local\uv\cache\archive-v0\eV2vg-8K3hBmoFg6\Lib\site-packages\torch\_subclasses\functional_tensor.py:368: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


Vocabulary : 62 tokens
  Lowercase: ['a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']
  Uppercase: ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z']
  Digits   : ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9']

Logit range: -4.0  to  4.0
Top 5 by logit:
  'e' : 4.0
  't' : 3.2
  'a' : 2.8
  'o' : 2.3
  'i' : 2.0


### The probability distribution

In [2]:
# input : toy_logits (62,)
# output: probability distribution as ASCII bar chart with surprisal
BAR_WIDTH   = 40
toy_probs   = F.softmax(toy_logits, dim=-1)
sorted_p, sorted_i = torch.sort(toy_probs, descending=True)

print(f"Probability distribution over {vocab_size} tokens (T=1.0)")
print(f"Showing all tokens with prob > 0.001")
print()
print(f"  {'Token':>6}  {'Logit':>6}  {'Prob':>7}  {'Surprisal':>10}  Bar")
print(f"  {'-'*68}")
for p, i in zip(sorted_p, sorted_i):
    prob = p.item()
    if prob < 0.001:
        break
    ch        = f"'{vocab_chars[i.item()]}'"
    logit     = toy_logits[i].item()
    surprisal = -math.log2(prob)
    bar       = chr(9608) * int(prob * BAR_WIDTH)
    print(f"  {ch:>6}  {logit:>6.1f}  {prob:>7.4f}  {surprisal:>10.3f}  {bar}")

print()
print(f"  Top  5 tokens cover : {sorted_p[:5].sum().item():.3f} of total mass")
print(f"  Top 10 tokens cover : {sorted_p[:10].sum().item():.3f} of total mass")
print(f"  Remaining {vocab_size-10:>2} tokens : {sorted_p[10:].sum().item():.3f} of total mass")

Probability distribution over 62 tokens (T=1.0)
Showing all tokens with prob > 0.001

   Token   Logit     Prob   Surprisal  Bar
  --------------------------------------------------------------------
     'e'     4.0   0.3972       1.332  ███████████████
     't'     3.2   0.1785       2.486  ███████
     'a'     2.8   0.1196       3.063  ████
     'o'     2.3   0.0726       3.785  ██
     'i'     2.0   0.0538       4.217  ██
     'n'     1.7   0.0398       4.650  █
     's'     1.4   0.0295       5.083  █
     'h'     1.1   0.0219       5.516  
     'r'     0.8   0.0162       5.949  
     'l'     0.5   0.0120       6.381  
     'd'     0.2   0.0089       6.814  
     'u'     0.0   0.0073       7.103  
     'c'    -0.2   0.0060       7.391  
     'm'    -0.4   0.0049       7.680  
     'f'    -0.6   0.0040       7.968  
     'w'    -0.8   0.0033       8.257  
     'g'    -1.0   0.0027       8.545  
     'p'    -1.2   0.0022       8.834  
     'y'    -1.4   0.0018       9.122  
     'b'

---
## Strategy 1 — Greedy

In [3]:
# input : toy_logits (62,)
# output: the single most probable token — argmax
# Greedy: always pick the highest-logit token. Fully deterministic.
# Pipeline: logits -> argmax -> token

greedy_idx  = torch.argmax(toy_logits).item()
greedy_prob = toy_probs[greedy_idx].item()

print(f"Greedy picks: '{vocab_chars[greedy_idx]}'")
print(f"  logit = {toy_logits[greedy_idx].item():.1f}")
print(f"  prob  = {greedy_prob:.4f}  ({greedy_prob*100:.1f}% of all mass)")
print(f"  surprisal = {-math.log2(greedy_prob):.3f} bits")
print()
print("Every other token is ignored — regardless of its probability.")
print()

# Show what greedy passes over
print("What greedy discards:")
for p, i in zip(sorted_p[1:6], sorted_i[1:6]):
    bar = chr(9608) * int(p.item() * BAR_WIDTH)
    print(f"  '{vocab_chars[i.item()]}' : {p.item():.4f}  {bar}  <- discarded")
print(f"  ... and {vocab_size-6} more tokens")
print()
print("Strength : deterministic, fast, always picks the highest-probability token.")
print("Weakness : zero diversity. Given the same input, output is always identical.")

Greedy picks: 'e'
  logit = 4.0
  prob  = 0.3972  (39.7% of all mass)
  surprisal = 1.332 bits

Every other token is ignored — regardless of its probability.

What greedy discards:
  't' : 0.1785  ███████  <- discarded
  'a' : 0.1196  ████  <- discarded
  'o' : 0.0726  ██  <- discarded
  'i' : 0.0538  ██  <- discarded
  'n' : 0.0398  █  <- discarded
  ... and 56 more tokens

Strength : deterministic, fast, always picks the highest-probability token.
Weakness : zero diversity. Given the same input, output is always identical.


---
## Strategy 2 — Temperature

In [4]:
# input : toy_logits (62,),  temperature T > 0
# output: reshaped distribution — higher T flattens, lower T sharpens
# Temperature: divide logits by T before softmax.
# T < 1  -> sharpens -> model more decisive
# T = 1  -> unmodified softmax
# T > 1  -> flattens -> model less decisive, more variety
# Pipeline: logits -> logits/T -> softmax -> multinomial sample

def temperature_sample(logits, temperature=1.0):
    probs = F.softmax(logits / temperature, dim=-1)
    return torch.multinomial(probs, num_samples=1).item()

temps = [0.5, 1.0, 2.0]
probs_by_temp = {t: F.softmax(toy_logits / t, dim=-1) for t in temps}

# Show top-10 tokens at each temperature as ASCII bars
print(f"Temperature effect on the distribution (top-10 tokens by T=1.0 rank)")
print()
top10_idx = torch.topk(toy_probs, 10).indices
for t in temps:
    p = probs_by_temp[t]
    print(f"  T = {t}")
    print(f"  {'-'*55}")
    for idx_t in top10_idx:
        ch   = f"'{vocab_chars[idx_t.item()]}'"
        prob = p[idx_t].item()
        bar  = chr(9608) * int(prob * BAR_WIDTH)
        print(f"  {ch:>4}  {prob:.4f}  {bar}")
    print()

# Show entropy at each temperature
print("Entropy (bits) — higher entropy = more uniform = more diversity:")
for t in temps:
    p   = probs_by_temp[t]
    ent = -(p * torch.log2(p + 1e-10)).sum().item()
    top = p.max().item()
    print(f"  T={t}  entropy={ent:.3f} bits   top-token prob={top:.4f}")
print()
print("Sampling 20 tokens at each temperature:")
for t in temps:
    torch.manual_seed(42)
    samples = [vocab_chars[temperature_sample(toy_logits, t)] for _ in range(20)]
    print(f"  T={t}: {samples}")

print()
print("Numerical stability note:")
print("  logits / T before softmax is safe in practice.")
print("  Modern implementations subtract max(logits) before exponentiation")
print("  (the max-subtraction trick), so extreme T values don't overflow.")
print("  T→0  : near-point-mass on the top token  (approaches greedy)")
print("  T→∞  : near-uniform over all vocab tokens (approaches random)")

Temperature effect on the distribution (top-10 tokens by T=1.0 rank)

  T = 0.5
  -------------------------------------------------------
   'e'  0.7315  █████████████████████████████
   't'  0.1477  █████
   'a'  0.0664  ██
   'o'  0.0244  
   'i'  0.0134  
   'n'  0.0074  
   's'  0.0040  
   'h'  0.0022  
   'r'  0.0012  
   'l'  0.0007  

  T = 1.0
  -------------------------------------------------------
   'e'  0.3972  ███████████████
   't'  0.1785  ███████
   'a'  0.1196  ████
   'o'  0.0726  ██
   'i'  0.0538  ██
   'n'  0.0398  █
   's'  0.0295  █
   'h'  0.0219  
   'r'  0.0162  
   'l'  0.0120  

  T = 2.0
  -------------------------------------------------------
   'e'  0.1536  ██████
   't'  0.1030  ████
   'a'  0.0843  ███
   'o'  0.0657  ██
   'i'  0.0565  ██
   'n'  0.0486  █
   's'  0.0419  █
   'h'  0.0360  █
   'r'  0.0310  █
   'l'  0.0267  █

Entropy (bits) — higher entropy = more uniform = more diversity:
  T=0.5  entropy=1.348 bits   top-token prob=0.7315
  T=1.

#### Watching randomness in action

In [5]:
# Pick one token at a time and repeat — notice different tokens get chosen at T > 1.
# Same seed each round so you can rerun and compare.
import torch, torch.nn.functional as F

print("Single picks from the toy distribution (10 trials per temperature)")
print()
for t in [0.5, 1.0, 2.0]:
    picks = []
    for trial in range(10):
        torch.manual_seed(trial)          # different seed each trial
        probs = F.softmax(toy_logits / t, dim=-1)
        idx   = torch.multinomial(probs, num_samples=1).item()
        picks.append(vocab_chars[idx])
    print(f"  T={t}  picks: {picks}")
    unique = set(picks)
    print(f"         {len(unique)} unique token(s) across 10 trials: {sorted(unique)}")
    # Flag any digits or uppercase that snuck in
    surprises = [ch for ch in picks if ch.isdigit() or ch.isupper()]
    if surprises:
        print(f"         ** note: {surprises} appeared — low-probability tokens winning because T={t} flattened the distribution")
    print()

Single picks from the toy distribution (10 trials per temperature)

  T=0.5  picks: ['e', 'a', 't', 'a', 'n', 'e', 't', 'e', 't', 'e']
         4 unique token(s) across 10 trials: ['a', 'e', 'n', 't']

  T=1.0  picks: ['e', 'a', 't', 'a', 'n', 'e', 't', 'n', 't', 'e']
         4 unique token(s) across 10 trials: ['a', 'e', 'n', 't']

  T=2.0  picks: ['6', 'a', 't', 'a', 'n', 'o', 't', 'n', 'h', 'e']
         7 unique token(s) across 10 trials: ['6', 'a', 'e', 'h', 'n', 'o', 't']
         ** note: ['6'] appeared — low-probability tokens winning because T=2.0 flattened the distribution



---
## Strategy 3 — Top-k

In [6]:
# input : toy_logits (62,),  k = number of candidates to keep
# output: sampled token from top-k candidates only
# Top-k: set all but the k highest logits to -inf, then softmax and sample.
# -inf -> 0 after softmax, so those tokens get zero probability.
# Pipeline: logits -> zero out all but top-k (set to -inf) -> softmax -> multinomial sample
# Weakness: k is fixed — does not adapt to how confident the model is.

def top_k_sample(logits, k):
    top_vals, top_idx = torch.topk(logits, k)
    filtered = torch.full_like(logits, float('-inf'))   # start: everything masked
    filtered[top_idx] = top_vals                         # restore only top-k scores
    probs = F.softmax(filtered, dim=-1)                  # -inf -> 0, top-k -> probabilities
    return torch.multinomial(probs, num_samples=1).item()

print("What top-k keeps and discards at different k values:")
print()
for k in [3, 5, 10]:
    top_vals, top_idx = torch.topk(toy_probs, k)
    kept   = [f"'{vocab_chars[i.item()]}'" for i in top_idx]
    mass   = top_vals.sum().item()
    cut_n  = vocab_size - k
    cut_m  = 1 - mass
    print(f"  k={k:<3}  keeps {k} tokens covering {mass:.3f} of mass: {kept}")
    print(f"        cuts  {cut_n} tokens -> set to -inf -> prob = 0.000")
    print()

print("The fixed-k problem:")
print("  A model may be near-certain (top token holds ~80% of mass) or uncertain")
print("  (top token holds ~5%) depending on context. Fixed k=5 always keeps")
print("  exactly 5 tokens regardless — too few when uncertain, too many when confident:")
print("  Confident (peaked): k=5 may keep near-zero tokens unnecessarily")
print("  Uncertain (flat)  : k=5 may cut tokens that should be candidates")
print()
print("20 samples at k=5:")
torch.manual_seed(42)
s = [vocab_chars[top_k_sample(toy_logits, k=5)] for _ in range(20)]
print(f"  {s}")

What top-k keeps and discards at different k values:

  k=3    keeps 3 tokens covering 0.695 of mass: ["'e'", "'t'", "'a'"]
        cuts  59 tokens -> set to -inf -> prob = 0.000

  k=5    keeps 5 tokens covering 0.822 of mass: ["'e'", "'t'", "'a'", "'o'", "'i'"]
        cuts  57 tokens -> set to -inf -> prob = 0.000

  k=10   keeps 10 tokens covering 0.941 of mass: ["'e'", "'t'", "'a'", "'o'", "'i'", "'n'", "'s'", "'h'", "'r'", "'l'"]
        cuts  52 tokens -> set to -inf -> prob = 0.000

The fixed-k problem:
  A model may be near-certain (top token holds ~80% of mass) or uncertain
  (top token holds ~5%) depending on context. Fixed k=5 always keeps
  exactly 5 tokens regardless — too few when uncertain, too many when confident:
  Confident (peaked): k=5 may keep near-zero tokens unnecessarily
  Uncertain (flat)  : k=5 may cut tokens that should be candidates

20 samples at k=5:
  ['a', 'e', 'e', 'e', 'e', 'a', 'i', 'e', 'a', 'e', 'e', 't', 'e', 'o', 'e', 'e', 'e', 'e', 'e', 't']


---
## Strategy 4 — Top-p (Nucleus Sampling)

In [7]:
# input : toy_logits (62,),  p = cumulative mass threshold
# output: sampled token from the smallest set of tokens covering p of the mass
# Top-p (nucleus): sort by probability, accumulate until cumsum >= p, sample from that set.
# Pipeline: logits -> softmax -> sort desc -> cumsum >= p -> zero rest -> renorm -> sample
# Advantage over top-k: nucleus size adapts to model confidence automatically.

def top_p_sample(logits, p):
    probs = F.softmax(logits, dim=-1)
    sorted_probs, sorted_idx = torch.sort(probs, descending=True)
    cumsum      = torch.cumsum(sorted_probs, dim=-1)
    remove_mask = (cumsum - sorted_probs) >= p   # True for tokens beyond the nucleus
    sorted_probs[remove_mask] = 0.0              # zero them out
    sorted_probs = sorted_probs / sorted_probs.sum()  # renorm survivors to sum=1
    pos = torch.multinomial(sorted_probs, num_samples=1)
    return sorted_idx[pos].item()

print("Nucleus size at different p values (this distribution):")
print()
sorted_p2, sorted_i2 = torch.sort(toy_probs, descending=True)
cumsum = torch.cumsum(sorted_p2, dim=-1)

for p_val in [0.5, 0.8, 0.9, 0.95]:
    n = int((cumsum <= p_val).sum().item()) + 1
    nucleus = [f"'{vocab_chars[sorted_i2[i].item()]}'" for i in range(n)]
    print(f"  p={p_val}  nucleus = {n:2d} tokens: {nucleus}")

print()
print("Compare to top-k=5 (always exactly 5, regardless of distribution shape):")
print("  top-k=5 : always 5 tokens")
print("  top-p=0.9: adapts — fewer when model is confident, more when uncertain")
print()
print("20 samples at p=0.9:")
torch.manual_seed(42)
s = [vocab_chars[top_p_sample(toy_logits, p=0.9)] for _ in range(20)]
print(f"  {s}")

Nucleus size at different p values (this distribution):

  p=0.5  nucleus =  2 tokens: ["'e'", "'t'"]
  p=0.8  nucleus =  5 tokens: ["'e'", "'t'", "'a'", "'o'", "'i'"]
  p=0.9  nucleus =  8 tokens: ["'e'", "'t'", "'a'", "'o'", "'i'", "'n'", "'s'", "'h'"]
  p=0.95  nucleus = 12 tokens: ["'e'", "'t'", "'a'", "'o'", "'i'", "'n'", "'s'", "'h'", "'r'", "'l'", "'d'", "'u'"]

Compare to top-k=5 (always exactly 5, regardless of distribution shape):
  top-k=5 : always 5 tokens
  top-p=0.9: adapts — fewer when model is confident, more when uncertain

20 samples at p=0.9:
  ['e', 'e', 'n', 't', 'a', 'e', 's', 'a', 'e', 's', 'i', 't', 'a', 'n', 'n', 'a', 'e', 't', 't', 'a']


---
## Strategy 5 — Min-p

In [8]:
# input : toy_logits (62,),  min_p = relative threshold
# output: sampled token where p(token) >= min_p * p(top_token)
# Min-p: keep any token at least min_p times as likely as the best token.
# Pipeline: logits -> softmax -> threshold = min_p * max_prob -> zero rest -> renorm -> sample
# Advantage: threshold scales automatically with the distribution — no calibration needed.

def min_p_sample(logits, min_p):
    probs     = F.softmax(logits, dim=-1)
    threshold = min_p * probs.max()
    filtered  = probs * (probs >= threshold)  # zero out tokens below threshold
    filtered  = filtered / filtered.sum()     # renorm survivors to sum=1
    return torch.multinomial(filtered, num_samples=1).item()

max_prob  = toy_probs.max().item()
top_char  = vocab_chars[toy_probs.argmax().item()]
print(f"Top token: '{top_char}'  prob = {max_prob:.4f}")
print()
print("Min-p thresholds (relative to top token):")
print()
for mp in [0.05, 0.1, 0.2, 0.5]:
    threshold = mp * max_prob
    kept = [(vocab_chars[i], p.item())
            for i, p in enumerate(toy_probs) if p.item() >= threshold]
    names = [f"'{t[0]}'" for t in kept]
    print(f"  min_p={mp}  threshold={threshold:.4f}  keeps {len(kept):2d} tokens: {names}")

print()
print("Contrast with top-p=0.9:")
n_top_p = int((torch.cumsum(torch.sort(toy_probs, descending=True).values, dim=-1) <= 0.9).sum().item()) + 1
n_min_p = sum(1 for p in toy_probs if p.item() >= 0.1 * max_prob)
print(f"  top-p=0.90 nucleus : {n_top_p} tokens (cumulative mass threshold)")
print(f"  min-p=0.10 nucleus : {n_min_p} tokens (relative-to-best threshold)")
print()
print("20 samples at min_p=0.1:")
torch.manual_seed(42)
s = [vocab_chars[min_p_sample(toy_logits, min_p=0.1)] for _ in range(20)]
print(f"  {s}")

print()
print("Min-p positioning:")
print("  Min-p is newer (Nguyen et al. 2024) and gaining adoption in inference engines.")
print("  Top-p remains the industry standard with more tuning guidance in the literature.")
print("  Min-p's relative threshold is conceptually cleaner for extreme confidence variation.")

Top token: 'e'  prob = 0.3972

Min-p thresholds (relative to top token):

  min_p=0.05  threshold=0.0199  keeps  8 tokens: ["'a'", "'e'", "'h'", "'i'", "'n'", "'o'", "'s'", "'t'"]
  min_p=0.1  threshold=0.0397  keeps  6 tokens: ["'a'", "'e'", "'i'", "'n'", "'o'", "'t'"]
  min_p=0.2  threshold=0.0794  keeps  3 tokens: ["'a'", "'e'", "'t'"]
  min_p=0.5  threshold=0.1986  keeps  1 tokens: ["'e'"]

Contrast with top-p=0.9:
  top-p=0.90 nucleus : 8 tokens (cumulative mass threshold)
  min-p=0.10 nucleus : 6 tokens (relative-to-best threshold)

20 samples at min_p=0.1:
  ['a', 'n', 'e', 'e', 'e', 'a', 'i', 'e', 'a', 'e', 'e', 't', 'e', 'o', 'n', 'e', 'e', 'e', 'e', 't']

Min-p positioning:
  Min-p is newer (Nguyen et al. 2024) and gaining adoption in inference engines.
  Top-p remains the industry standard with more tuning guidance in the literature.
  Min-p's relative threshold is conceptually cleaner for extreme confidence variation.


---
## Part 2 — Transfer to a Trained Model

The five strategy functions above were built on a hand-crafted distribution. We now load tinyshakespeare, train a bigram model, obtain real logits, and run the same functions unchanged. The only difference is where the logits come from.

### Load corpus and train bigram model

In [9]:
# input : tinyshakespeare.txt
# output: trained BigramModel + logits for context 'T'
import torch.nn as nn

with open('tinyshakespeare.txt', 'r', encoding='utf-8') as f:
    text = f.read()

real_chars      = sorted(set(text))
real_vocab_size = len(real_chars)
real_stoi       = {ch: i for i, ch in enumerate(real_chars)}
real_itos       = {i: ch for i, ch in enumerate(real_chars)}
encode          = lambda s: [real_stoi[c] for c in s]
decode          = lambda l: ''.join([real_itos[i] for i in l])
data            = torch.tensor(encode(text), dtype=torch.long)
device          = 'cuda' if torch.cuda.is_available() else 'cpu'

class BigramModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.embedding_table = nn.Embedding(vocab_size, vocab_size)
    def forward(self, idx, targets=None):
        logits = self.embedding_table(idx)
        loss   = None
        if targets is not None:
            B, T, C = logits.shape
            loss = F.cross_entropy(logits.view(B*T, C), targets.view(B*T))
        return logits, loss

def get_batch(data, block_size, batch_size, device='cpu'):
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x  = torch.stack([data[i  :i+block_size  ] for i in ix])
    y  = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x.to(device), y.to(device)

torch.manual_seed(42)
model     = BigramModel(real_vocab_size).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-2)
for _ in range(3000):
    xb, yb = get_batch(data, 4, 32, device)
    _, loss = model(xb, yb)
    optimizer.zero_grad(); loss.backward(); optimizer.step()

# Get logits for context 'T' — same character used in Part 1 toy
context = torch.tensor([encode('T')], dtype=torch.long).to(device)
with torch.no_grad():
    all_logits, _ = model(context)
    real_logits   = all_logits[0, -1, :]     # (real_vocab_size,)

print(f"Corpus        : {len(text):,} characters, {real_vocab_size} unique tokens")
print(f"Model trained : BigramModel  |  final loss = {loss.item():.4f}")
print(f"Real logits   : shape {real_logits.shape}  (context: 'T')")

Corpus        : 1,115,394 characters, 65 unique tokens
Model trained : BigramModel  |  final loss = 2.4349
Real logits   : shape torch.Size([65])  (context: 'T')


---
## Real distribution after 'T'

In [10]:
# input : real_logits (real_vocab_size,)
# output: top-5 and top-10 summary of the real distribution
real_probs = F.softmax(real_logits, dim=-1)
real_sorted_p, real_sorted_i = torch.sort(real_probs, descending=True)

print(f"Real distribution after 'T'  ({real_vocab_size} tokens total)")
print()
for n in [5, 10]:
    mass = real_sorted_p[:n].sum().item()
    tokens = [repr(real_itos[real_sorted_i[i].item()]) for i in range(n)]
    print(f"  Top {n:>2}: {tokens}")
    print(f"          covers {mass:.3f} of total mass")
    print()

print("Entropy:", round(-(real_probs * torch.log2(real_probs + 1e-10)).sum().item(), 3), "bits")

Real distribution after 'T'  (65 tokens total)

  Top  5: ["'h'", "'o'", "'E'", "'I'", "':'"]
          covers 0.740 of total mass

  Top 10: ["'h'", "'o'", "'E'", "'I'", "':'", "'R'", "'i'", "'H'", "'a'", "'A'"]
          covers 0.872 of total mass

Entropy: 3.264 bits


### Greedy on the real model

In [11]:
# Same argmax rule — now applied to the trained model in a generation loop.
# The bigram model only uses the last token, so the loop locks into a cycle quickly.

def generate_greedy(model, start_ids, max_tokens):
    idx = torch.tensor([start_ids], dtype=torch.long).to(device)
    for _ in range(max_tokens):
        with torch.no_grad():
            logits_g, _ = model(idx)
        next_tok = torch.argmax(logits_g[:, -1, :], dim=-1, keepdim=True)
        idx = torch.cat([idx, next_tok], dim=1)
    return decode(idx[0].tolist())

print("Greedy step-by-step from 'T' (first 15 steps):")
print(f"  {'Step':>4}  {'Input':>6}  {'Predicted':>10}  {'Prob':>7}")
print("  " + "-" * 34)

ctx = encode('T')
idx = torch.tensor([ctx], dtype=torch.long).to(device)
for step in range(15):
    with torch.no_grad():
        logits_g, _ = model(idx)
    p        = F.softmax(logits_g[0, -1, :], dim=-1)
    next_tok = torch.argmax(p).item()
    in_ch    = repr(real_itos[idx[0, -1].item()])
    out_ch   = repr(real_itos[next_tok])
    print(f"  {step+1:>4}  {in_ch:>6}  {out_ch:>10}  {p[next_tok].item():>7.4f}")
    idx = torch.cat([idx, torch.tensor([[next_tok]]).to(device)], dim=1)

print()
print("Full greedy generation (40 tokens):")
print(f"  '{generate_greedy(model, encode('T'), 40)}'")
print()
print("The cycle is visible in the step trace above — once entered it never breaks.")

Greedy step-by-step from 'T' (first 15 steps):
  Step   Input   Predicted     Prob
  ----------------------------------
     1     'T'         'h'   0.3948
     2     'h'         'e'   0.3442
     3     'e'         ' '   0.2919
     4     ' '         't'   0.1471
     5     't'         'h'   0.3304
     6     'h'         'e'   0.3442
     7     'e'         ' '   0.2919
     8     ' '         't'   0.1471
     9     't'         'h'   0.3304
    10     'h'         'e'   0.3442
    11     'e'         ' '   0.2919
    12     ' '         't'   0.1471
    13     't'         'h'   0.3304
    14     'h'         'e'   0.3442
    15     'e'         ' '   0.2919

Full greedy generation (40 tokens):
  'The the the the the the the the the the t'

The cycle is visible in the step trace above — once entered it never breaks.


### Temperature on the real model

In [12]:
# Same temperature_sample function — now applied to real model logits.

def generate_temperature(model, start_ids, max_tokens, temperature=1.0):
    idx = torch.tensor([start_ids], dtype=torch.long).to(device)
    for _ in range(max_tokens):
        with torch.no_grad():
            logits_t, _ = model(idx)
        next_tok = temperature_sample(logits_t[0, -1, :], temperature)
        idx = torch.cat([idx, torch.tensor([[next_tok]]).to(device)], dim=1)
    return decode(idx[0].tolist())

# Bar chart: real distribution at three temperatures
probs_by_temp_real = {t: F.softmax(real_logits / t, dim=-1) for t in [0.5, 1.0, 2.0]}
top10_real = torch.topk(real_probs, 10).indices

print("Temperature effect on real distribution (top-10 by T=1.0 rank):")
print()
for t in [0.5, 1.0, 2.0]:
    p = probs_by_temp_real[t]
    print(f"  T = {t}")
    print(f"  {'-'*52}")
    for idx_t in top10_real:
        ch   = repr(real_itos[idx_t.item()])
        prob = p[idx_t].item()
        bar  = chr(9608) * int(prob * BAR_WIDTH)
        print(f"  {ch:>5}  {prob:.4f}  {bar}")
    print()

print("Generation at different temperatures (40 tokens from 'T'):")
for t in [0.5, 1.0, 2.0]:
    torch.manual_seed(42)
    out = generate_temperature(model, encode('T'), 40, temperature=t)
    print(f"  T={t}: '{out}'")

Temperature effect on real distribution (top-10 by T=1.0 rank):

  T = 0.5
  ----------------------------------------------------
    'h'  0.8014  ████████████████████████████████
    'o'  0.0951  ███
    'E'  0.0298  █
    'I'  0.0248  
    ':'  0.0211  
    'R'  0.0063  
    'i'  0.0046  
    'H'  0.0046  
    'a'  0.0018  
    'A'  0.0016  

  T = 1.0
  ----------------------------------------------------
    'h'  0.3948  ███████████████
    'o'  0.1360  █████
    'E'  0.0761  ███
    'I'  0.0694  ██
    ':'  0.0641  ██
    'R'  0.0351  █
    'i'  0.0299  █
    'H'  0.0299  █
    'a'  0.0188  
    'A'  0.0176  

  T = 2.0
  ----------------------------------------------------
    'h'  0.1460  █████
    'o'  0.0857  ███
    'E'  0.0641  ██
    'I'  0.0612  ██
    ':'  0.0588  ██
    'R'  0.0435  █
    'i'  0.0402  █
    'H'  0.0402  █
    'a'  0.0319  █
    'A'  0.0309  █

Generation at different temperatures (40 tokens from 'T'):
  T=0.5: 'Thisthoul shathis nd m arer we an, thathe'


### Top-k on the real model

In [13]:
# Same top_k_sample function.
print("Top-k nucleus on the real distribution:")
print()
for k in [3, 5, 10]:
    top_vals, top_idx = torch.topk(real_probs, k)
    kept = [repr(real_itos[i.item()]) for i in top_idx]
    mass = top_vals.sum().item()
    print(f"  k={k:<3}  covers {mass:.3f} of mass: {kept}")
    print(f"        {real_vocab_size - k} tokens set to -inf -> prob = 0.000")

print()
print("Fixed-k weakness: k=5 always keeps 5 tokens regardless of confidence.")
print()
print("20 samples (k=5):")
torch.manual_seed(42)
s = [real_itos[top_k_sample(real_logits, k=5)] for _ in range(20)]
print(f"  {s}")

Top-k nucleus on the real distribution:

  k=3    covers 0.607 of mass: ["'h'", "'o'", "'E'"]
        62 tokens set to -inf -> prob = 0.000
  k=5    covers 0.740 of mass: ["'h'", "'o'", "'E'", "'I'", "':'"]
        60 tokens set to -inf -> prob = 0.000
  k=10   covers 0.872 of mass: ["'h'", "'o'", "'E'", "'I'", "':'", "'R'", "'i'", "'H'", "'a'", "'A'"]
        55 tokens set to -inf -> prob = 0.000

Fixed-k weakness: k=5 always keeps 5 tokens regardless of confidence.

20 samples (k=5):
  ['h', ':', 'h', 'h', 'h', 'o', 'h', 'o', 'o', 'h', 'h', ':', 'h', 'h', 'o', 'o', 'h', ':', 'h', 'h']


### Top-p on the real model

In [14]:
# Same top_p_sample function.
real_sorted_p2, real_sorted_i2 = torch.sort(real_probs, descending=True)
real_cumsum = torch.cumsum(real_sorted_p2, dim=-1)

print("Nucleus size adapts to real model confidence:")
for p_val in [0.5, 0.8, 0.9, 0.95]:
    n = int((real_cumsum <= p_val).sum().item()) + 1
    nucleus = [repr(real_itos[real_sorted_i2[i].item()]) for i in range(min(n, 8))]
    tail = ' ...' if n > 8 else ''
    print(f"  p={p_val}  nucleus = {n:2d} tokens: {nucleus}{tail}")

print()
print("20 samples (p=0.9):")
torch.manual_seed(42)
s = [real_itos[top_p_sample(real_logits, p=0.9)] for _ in range(20)]
print(f"  {s}")

Nucleus size adapts to real model confidence:
  p=0.5  nucleus =  2 tokens: ["'h'", "'o'"]
  p=0.8  nucleus =  7 tokens: ["'h'", "'o'", "'E'", "'I'", "':'", "'R'", "'i'"]
  p=0.9  nucleus = 12 tokens: ["'h'", "'o'", "'E'", "'I'", "':'", "'R'", "'i'", "'H'"] ...
  p=0.95  nucleus = 15 tokens: ["'h'", "'o'", "'E'", "'I'", "':'", "'R'", "'i'", "'H'"] ...

20 samples (p=0.9):
  ['h', 'U', ':', 'h', 'h', 'h', 'o', 'o', 'o', ':', 'h', 'o', 'h', 'h', 'i', 'h', 'i', 'h', 'I', 'o']


### Min-p on the real model

In [15]:
# Same min_p_sample function.
real_max_prob = real_probs.max().item()
real_top_ch   = real_itos[real_probs.argmax().item()]
print(f"Top token: '{real_top_ch}'  prob = {real_max_prob:.4f}")
print()
print("Min-p thresholds on real distribution:")
for mp in [0.05, 0.1, 0.2]:
    threshold = mp * real_max_prob
    kept = [repr(real_itos[i]) for i, p in enumerate(real_probs) if p.item() >= threshold]
    print(f"  min_p={mp}  threshold={threshold:.4f}  keeps {len(kept)} tokens: {kept}")

print()
print("20 samples (min_p=0.1):")
torch.manual_seed(42)
s = [real_itos[min_p_sample(real_logits, min_p=0.1)] for _ in range(20)]
print(f"  {s}")

Top token: 'h'  prob = 0.3948

Min-p thresholds on real distribution:
  min_p=0.05  threshold=0.0197  keeps 8 tokens: ["':'", "'E'", "'H'", "'I'", "'R'", "'h'", "'i'", "'o'"]
  min_p=0.1  threshold=0.0395  keeps 5 tokens: ["':'", "'E'", "'I'", "'h'", "'o'"]
  min_p=0.2  threshold=0.0790  keeps 2 tokens: ["'h'", "'o'"]

20 samples (min_p=0.1):
  ['h', ':', 'h', 'h', 'h', 'o', 'h', 'o', 'o', 'h', 'h', ':', 'h', 'h', 'o', 'o', 'h', ':', 'h', 'h']


---
## The Full Decoding Pipeline

Each strategy so far controlled one aspect of sampling. In practice they compose:
temperature reshapes the distribution first, then a truncation strategy restricts
the candidates, then multinomial draws one token.

```
logits
  → divide by temperature
  → softmax
  → truncate with top-p / top-k / min-p
  → renormalize
  → multinomial sample
  → next token
```

In [16]:
# Combined pipeline: temperature + truncation strategy + multinomial sample.
# Each stage is independent; compose any combination.

def sample(logits, temperature=1.0, top_p=1.0, top_k=None, min_p=None):
    # Stage 1 — temperature: reshape the distribution
    probs = F.softmax(logits / temperature, dim=-1)

    # Stage 2 — truncation: restrict candidates (pick one strategy)
    if top_k is not None:
        top_vals, top_idx = torch.topk(probs, top_k)
        filtered = torch.zeros_like(probs)
        filtered[top_idx] = top_vals
        probs = filtered
    elif min_p is not None:
        threshold = min_p * probs.max()
        probs = probs * (probs >= threshold)
    elif top_p < 1.0:
        sorted_probs, sorted_idx = torch.sort(probs, descending=True)
        cumsum = torch.cumsum(sorted_probs, dim=-1)
        remove_mask = (cumsum - sorted_probs) >= top_p
        sorted_probs[remove_mask] = 0.0
        probs = torch.zeros_like(probs)
        probs[sorted_idx] = sorted_probs

    # Stage 3 — renormalize and draw
    probs = probs / probs.sum()
    return torch.multinomial(probs, num_samples=1).item()

# Show candidate count and top-token probability for each combination
print("Combined pipeline on the toy distribution (single draw, seed=42)")
print()
print(f"  {'Configuration':<30}  {'Candidates':>10}  {'Top-1 prob':>10}  Sample")
print("  " + "-" * 62)
configs = [
    ("greedy (T=0.0)",        dict(temperature=0.01)),
    ("T=1.0 only",            dict(temperature=1.0)),
    ("T=0.5 only",            dict(temperature=0.5)),
    ("T=1.0 + top-k=5",      dict(temperature=1.0, top_k=5)),
    ("T=0.5 + top-k=5",      dict(temperature=0.5, top_k=5)),
    ("T=1.0 + top-p=0.9",    dict(temperature=1.0, top_p=0.9)),
    ("T=0.5 + top-p=0.9",    dict(temperature=0.5, top_p=0.9)),
    ("T=1.0 + min-p=0.1",    dict(temperature=1.0, min_p=0.1)),
]
for label, kwargs in configs:
    p = F.softmax(toy_logits / kwargs.get('temperature', 1.0), dim=-1)
    if kwargs.get('top_k'):
        n = kwargs['top_k']
    elif kwargs.get('min_p'):
        threshold = kwargs['min_p'] * p.max()
        n = int((p >= threshold).sum().item())
    elif kwargs.get('top_p', 1.0) < 1.0:
        sp, _ = torch.sort(p, descending=True)
        cs = torch.cumsum(sp, dim=-1)
        n = int((cs <= kwargs['top_p']).sum().item()) + 1
    else:
        n = vocab_size
    torch.manual_seed(42)
    tok = vocab_chars[sample(toy_logits, **kwargs)]
    print(f"  {label:<30}  {n:>10}  {p.max().item():>10.4f}  '{tok}'")

Combined pipeline on the toy distribution (single draw, seed=42)

  Configuration                   Candidates  Top-1 prob  Sample
  --------------------------------------------------------------
  greedy (T=0.0)                          62      1.0000  'e'
  T=1.0 only                              62      0.3972  'a'
  T=0.5 only                              62      0.7315  'a'
  T=1.0 + top-k=5                          5      0.3972  'a'
  T=0.5 + top-k=5                          5      0.7315  'a'
  T=1.0 + top-p=0.9                        8      0.3972  'a'
  T=0.5 + top-p=0.9                        3      0.7315  'a'
  T=1.0 + min-p=0.1                        6      0.3972  'a'


---
## When to Use Which Strategy

| Strategy | Use when | Watch out for |
|---|---|---|
| **Greedy** | Reproducible, deterministic output — structured extraction, controlled experiments, evals | Repetition loops; zero diversity |
| **Temperature** | Controlling creative risk — more conservative vs more exploratory | High T samples rare tokens; low T approaches greedy |
| **Top-k** | Hard cap on candidates; stable distributions | Fixed k does not adapt to model confidence |
| **Top-p** | Adaptive nucleus for chat and creative tasks; standard default | Needs calibration per model |
| **Min-p** | Relative threshold that scales with top-token confidence | Less widely supported in APIs than top-p |

---

### Combining strategies

Temperature and a truncation strategy are almost always used together — they control different things:

- **Temperature** reshapes the distribution (how peaked or flat)
- **Top-p / top-k / min-p** restricts which tokens can be sampled from that reshaped distribution

The right combination is application and model dependent. What is stable across providers is the conceptual separation: temperature first, truncation second, sample third.

---

> **Core insight.** Every strategy in this chapter trades some of greedy's commitment to the locally highest-probability token for flexibility. Greedy commits fully at every step; temperature, top-k, top-p, and min-p all introduce uncertainty to avoid locally optimal sequences that lead into repetition cycles or dead ends. The choice of strategy is always about this trade-off: reliability vs. diversity.

---
## Summary — All Strategies

In [17]:
print("=" * 74)
print("SAMPLING STRATEGIES — SUMMARY")
print("=" * 74)
rows = [
    ("Greedy",       "logits -> argmax",                          "None",  "Yes"),
    ("Temperature",  "logits/T -> softmax -> sample",             "T",     "No"),
    ("Top-k",        "set -inf -> softmax -> sample",             "k",     "No"),
    ("Top-p",        "softmax -> cumsum>=p -> renorm -> sample",  "p",     "No"),
    ("Min-p",        "softmax -> keep>=min_p*max -> sample",      "min_p", "No"),
]
print(f"  {'Strategy':<14} {'Pipeline':<45} {'Controls':<8} {'Deterministic'}")
print("  " + "-" * 74)
for name, pipeline, controls, det in rows:
    print(f"  {name:<14} {pipeline:<45} {controls:<8} {det}")
print()
print("Combine temperature + any truncation strategy (see combined pipeline above):")
print("  temperature first, truncation second, sample third")

SAMPLING STRATEGIES — SUMMARY
  Strategy       Pipeline                                      Controls Deterministic
  --------------------------------------------------------------------------
  Greedy         logits -> argmax                              None     Yes
  Temperature    logits/T -> softmax -> sample                 T        No
  Top-k          set -inf -> softmax -> sample                 k        No
  Top-p          softmax -> cumsum>=p -> renorm -> sample      p        No
  Min-p          softmax -> keep>=min_p*max -> sample          min_p    No

Combine temperature + any truncation strategy (see combined pipeline above):
  temperature first, truncation second, sample third


---
## PyTorch Lab — Chapter 5

### Exercise 1 — Temperature and entropy

In [18]:
# Exercise 1: compute entropy of the toy distribution at T = 0.1, 0.5, 1.0, 2.0, 5.0.
# Entropy H = -sum(p * log2(p))  (bits).  Higher H = more uniform = more diversity.
print("Exercise 1: Temperature vs entropy (toy distribution)")
print(f"  {'T':>6}  {'Entropy (bits)':>16}  {'Top-1 prob':>12}")
print("  " + "-" * 38)
for t in [0.1, 0.5, 1.0, 2.0, 5.0]:
    p   = F.softmax(toy_logits / t, dim=-1)
    H   = -(p * torch.log2(p + 1e-10)).sum().item()
    top = p.max().item()
    print(f"  {t:>6.1f}  {H:>16.4f}  {top:>12.4f}")

print()
print("Note: entropy measures the shape of the distribution, not sample variance.")
print("A distribution with H=5 bits drawn once may still land on the top token.")
print("Realized diversity depends on both entropy AND the number of draws.")
print("The uniqueness counts in the temperature demo above show what happens")
print("empirically across 10 trials — that is what matters in practice.")

Exercise 1: Temperature vs entropy (toy distribution)
       T    Entropy (bits)    Top-1 prob
  --------------------------------------
     0.1            0.0045        0.9997
     0.5            1.3482        0.7315
     1.0            2.9797        0.3972
     2.0            4.8690        0.1536
     5.0            5.8078        0.0472

Note: entropy measures the shape of the distribution, not sample variance.
A distribution with H=5 bits drawn once may still land on the top token.
Realized diversity depends on both entropy AND the number of draws.
The uniqueness counts in the temperature demo above show what happens
empirically across 10 trials — that is what matters in practice.


### Exercise 2 — Implement top-p from scratch

In [19]:
# Exercise 2: implement nucleus sampling without using the reference version above.
def my_top_p_sample(logits, p, temperature=1.0):
    # TODO: implement
    # 1. scale by temperature, compute softmax
    # 2. sort descending
    # 3. cumulative sum
    # 4. mask tokens after cumsum crosses p
    # 5. renormalise and sample
    raise NotImplementedError("implement me")

try:
    torch.manual_seed(0)
    samples = [my_top_p_sample(toy_logits, p=0.9) for _ in range(100)]
    assert all(0 <= s < vocab_size for s in samples)
    print(f"Exercise 2: 100 samples OK -> {[vocab_chars[s] for s in samples[:20]]}")
except NotImplementedError:
    print("Exercise 2: not yet implemented.")

Exercise 2: not yet implemented.


### Exercise 3 — Top-k vs top-p under confidence extremes

In [20]:
# Exercise 3: construct peaked and flat toy distributions.
# Show how many tokens top-k=5 vs top-p=0.9 keep in each case.
peaked = torch.zeros(vocab_size); peaked[0] = 10.0
flat   = torch.zeros(vocab_size)

print("Exercise 3: top-k vs top-p under confidence extremes")
print()
for name, lgt in [("Peaked (one token dominates)", peaked), ("Flat (uniform)", flat)]:
    p_f = F.softmax(lgt, dim=-1)
    s_p, _ = torch.sort(p_f, descending=True)
    cs  = torch.cumsum(s_p, dim=-1)
    n_p = int((cs <= 0.9).sum().item()) + 1
    print(f"  {name}")
    print(f"    top-k=5  : always 5 tokens")
    print(f"    top-p=0.9: {n_p} tokens")
    print()

Exercise 3: top-k vs top-p under confidence extremes

  Peaked (one token dominates)
    top-k=5  : always 5 tokens
    top-p=0.9: 1 tokens

  Flat (uniform)
    top-k=5  : always 5 tokens
    top-p=0.9: 56 tokens



### Exercise 4 — Top-k vs top-p: which tokens do they agree on?

In [21]:
# Exercise 4: compare top-k and top-p on the real distribution.
# For k=5 and p=0.9, print which tokens each strategy keeps and whether they agree.
top_k_kept = set(torch.topk(real_probs, 5).indices.tolist())

real_s, real_i = torch.sort(real_probs, descending=True)
cs = torch.cumsum(real_s, dim=-1)
n  = int((cs <= 0.9).sum().item()) + 1
top_p_kept = set(real_i[:n].tolist())

print("Exercise 4: top-k=5 vs top-p=0.9 on real distribution")
print()
print(f"  top-k keeps : {[repr(real_itos[i]) for i in torch.topk(real_probs,5).indices.tolist()]}")
print(f"  top-p keeps : {[repr(real_itos[i]) for i in real_i[:n].tolist()]}")
print()
in_both   = top_k_kept & top_p_kept
only_topk = top_k_kept - top_p_kept
only_topp = top_p_kept - top_k_kept
print(f"  In both     : {[repr(real_itos[i]) for i in in_both]}")
print(f"  Only top-k  : {[repr(real_itos[i]) for i in only_topk]}")
print(f"  Only top-p  : {[repr(real_itos[i]) for i in only_topp]}")

Exercise 4: top-k=5 vs top-p=0.9 on real distribution

  top-k keeps : ["'h'", "'o'", "'E'", "'I'", "':'"]
  top-p keeps : ["'h'", "'o'", "'E'", "'I'", "':'", "'R'", "'i'", "'H'", "'a'", "'A'", "'U'", "'r'"]

  In both     : ["':'", "'h'", "'E'", "'o'", "'I'"]
  Only top-k  : []
  Only top-p  : ["'U'", "'a'", "'A'", "'i'", "'H'", "'r'", "'R'"]


### Exercise 5 — Strategy comparison on the same prompt

In [22]:
# Exercise 5: run greedy, T=1.0, and T=2.0 on the same prompt. Compare outputs.
prompt = encode('To be or not')
print("Exercise 5: strategy comparison ('To be or not', 40 tokens)")
print()
torch.manual_seed(42)
print(f"  Greedy : '{generate_greedy(model, prompt, 40)}'")
torch.manual_seed(42)
print(f"  T=1.0  : '{generate_temperature(model, prompt, 40, temperature=1.0)}'")
torch.manual_seed(42)
print(f"  T=2.0  : '{generate_temperature(model, prompt, 40, temperature=2.0)}'")

Exercise 5: strategy comparison ('To be or not', 40 tokens)

  Greedy : 'To be or nothe the the the the the the the the the t'
  T=1.0  : 'To be or not istloulfe
Thaday:
we my char nd, onoutn'
  T=2.0  : 'To be or not isDYOLMfe
T:
Cay:
we &IAU: G n.
tozoWhn'


---
## Research Connections

**Holtzman et al. (2019) — *The Curious Case of Neural Text Degeneration***
Introduced top-p (nucleus) sampling and diagnosed why sampling from the full softmax distribution produces degenerate text. The key finding: sampling probability is not fixed across steps — the distribution varies in shape with model confidence, and a fixed-k threshold fails to track that variation. Top-p constructs an adaptive nucleus that scales with the distribution.

**Nguyen et al. (2024) — *Min-p Sampling***
Proposed the relative-threshold alternative to top-p. The cutoff `p(token) >= min_p * p(top_token)` adapts naturally to the scale of the distribution without calibration. The paper showed improvements over top-p on creative generation benchmarks, particularly for models with widely varying confidence across steps.

**Sutskever, Vinyals & Le (2014) — *Sequence to Sequence Learning with Neural Networks***
Used beam search to approximately find high-probability output sequences in neural machine translation, illustrating an important distinction from greedy decoding: instead of committing to one locally best token at every step, beam search preserves multiple candidate continuations and scores them jointly.

---
## What This Chapter Deliberately Does Not Explain

**Beam search.** Maintains B candidate sequences in parallel and keeps the B highest-scoring ones at each step — a search strategy, not a sampling strategy, and deterministic given B. Beam search pays off when the model has real context memory (RNN, transformer) where different paths through the sequence lead to genuinely different outcomes. On a bigram model every beam collapses to the same repetition cycle immediately and B=1 (greedy) always wins. Covered properly in the sequence-to-sequence chapter when we have a model whose predictions actually depend on more than one token of history.

**Repetition penalties.** Many APIs subtract a penalty from the logits of recently-seen tokens to discourage loops — a post-hoc fix for the greedy/low-temperature problem. Useful in practice; not a sampling strategy in the strict sense.

**Typical sampling.** Meister et al. (2023) proposed sampling near the *typical* probability rather than the highest, motivated by information-theoretic arguments. Interesting but not widely deployed.

**Speculative decoding.** A draft model generates several tokens cheaply; the main model verifies them in one forward pass. A latency optimisation, not a sampling strategy. Covered in Book 2.

**Constrained decoding.** Grammars and finite-state machines that restrict valid tokens at each step — important for structured output (JSON, SQL, code). Covered in the text-to-SQL chapter.

---
*Next: Chapter 6 — Context Windows: Learning from More Than One Token.*
*The bigram model only looks at the last character. Everything it gets wrong traces to that limitation. Chapter 6 extends the context window and builds the first model that can actually remember a sentence.*

[^notebook]: *The notebook `ch05_sampling.ipynb` is the companion to this chapter ([github.com/asbassan/AmarLMFP](https://github.com/asbassan/AmarLMFP/blob/main/ch05_sampling.ipynb)). Run cells in order.*